# Phase 01: Prototype OCR Extraction

**Plan:** 01-01 — Prototyping OCR extraction and pdf2image loading

This notebook prototypes the end-to-end PDF → Image → OCR → Structured Text pipeline
on a small sample (3 PDFs) from Uthai Thani Constituency 2, running all three engines
side-by-side before scaling up in Phase 2.

**Target form:** ส.ส.5/18 (election day voting results per polling station)

**OCR Engines Under Test (in priority order per EXTR-04):**
| Priority | Engine | Mode |
|----------|--------|------|
| 1 | Typhoon OCR | API via `typhoon-ocr` package + `TYPHOON_OCR_API_KEY` |
| 2 | PaddleOCR | Local, `lang='th'` |
| 3 | Tesseract | Local, `lang=tha+eng` via `pytesseract` |

**PDF rendering:** PyMuPDF (fitz) at 200 DPI — no poppler dependency required.

## Cell 1: Environment Imports

In [10]:
import os
import re
import io
from pathlib import Path

import tempfile
import fitz  # PyMuPDF — PDF to image conversion
from PIL import Image
import pytesseract
from paddleocr import PaddleOCR
from typhoon_ocr import ocr_document

# Load .env for API keys
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('dotenv loaded')
except ImportError:
    print('python-dotenv not installed; skipping .env load')

print(f'PyMuPDF version: {fitz.version}')
print(f'Tesseract version: {pytesseract.get_tesseract_version()}')
print('All imports OK')

dotenv loaded
PyMuPDF version: ('1.27.2.2', '1.27.2', None)
Tesseract version: 5.3.4
All imports OK


## Cell 2: Configuration

In [ ]:
# Root directory of source PDFs (relative to notebook location)
NOTEBOOK_DIR = Path().resolve()
SOURCE_DIR = Path(os.getenv("SOURCE_DIR", str(NOTEBOOK_DIR.parent / "data" / "เขตเลือกตั้งที่ 2")))

print(f"SOURCE_DIR: {SOURCE_DIR}")
print(f"SOURCE_DIR exists: {SOURCE_DIR.exists()}")

# OCR settings
PDF_DPI = 200
SAMPLE_SIZE = 3
TARGET_FORM = "ส.ส.5ทับ18.pdf"

# Typhoon OCR settings
# api.openthai.ai is the correct endpoint for openthai API keys.
# The typhoon-ocr package defaults to api.opentyphoon.ai (different key space)
# and will 408-timeout with openthai keys.
TYPHOON_API_KEY = os.getenv("TYPHOON_OCR_API_KEY")
TYPHOON_BASE_URL = os.getenv("TYPHOON_OCR_BASE_URL", "https://api.openthai.ai/v1")

# Warn if .env file is missing
env_path = NOTEBOOK_DIR.parent / ".env"
if not env_path.exists():
    print(f"WARNING: .env not found — copy .env.example to .env and set your API key")

print(f"Typhoon API key set: {bool(TYPHOON_API_KEY)}")
print(f"Typhoon base URL:    {TYPHOON_BASE_URL}")
print(f"Target form: {TARGET_FORM}")
print(f"DPI: {PDF_DPI}, Sample size: {SAMPLE_SIZE}")

SOURCE_DIR: /home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/เขตเลือกตั้งที่ 2
SOURCE_DIR exists: True
Typhoon API key set: True
Typhoon base URL:    https://api.opentyphoon.ai/v1
Target form: ส.ส.5ทับ18.pdf
DPI: 200, Sample size: 3


## Cell 3: OCR Engine Initialization

In [12]:
# --- PaddleOCR (PP-OCRv5, local) ---
print('Initializing PaddleOCR (PP-OCRv5)...')
paddle_ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    lang='th'
)
print('PaddleOCR initialized')

# --- Tesseract (local) ---
print(f'Tesseract: {pytesseract.get_tesseract_version()}')
langs = pytesseract.get_languages()
has_thai = 'tha' in langs
print(f'Tesseract Thai lang data installed: {has_thai}')
if not has_thai:
    print('  Install with: sudo apt install tesseract-ocr-tha')

# --- Typhoon OCR (API) ---
if TYPHOON_API_KEY:
    print('Typhoon OCR API key: set')
else:
    print('WARNING: TYPHOON_OCR_API_KEY not set — Typhoon OCR cells will be skipped')

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/chatrin/.paddlex/official_models/PP-OCRv5_server_det`.


Initializing PaddleOCR (PP-OCRv5)...


Creating model: ('th_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/chatrin/.paddlex/official_models/th_PP-OCRv5_mobile_rec`.


PaddleOCR initialized
Tesseract: 5.3.4
Tesseract Thai lang data installed: True
Typhoon OCR API key: set


---
## Task 2: PDF Directory Traversal

In [13]:
def find_election_day_pdfs(root_dir: Path, target_filename: str = TARGET_FORM) -> list[dict]:
    """
    Recursively locate election-day PDF forms under the constituency directory.

    Expected structure:
      root_dir/
        {district}/
          {subdistrict}/
            {station}/
              ส.ส.5ทับ18.pdf

    Returns list of dicts: path, district, subdistrict, station
    """
    results = []
    for dirpath, _, filenames in os.walk(root_dir):
        if target_filename in filenames:
            dirpath = Path(dirpath)
            rel_parts = dirpath.relative_to(root_dir).parts
            results.append({
                'path': str(dirpath / target_filename),
                'district':    rel_parts[0] if len(rel_parts) > 0 else 'unknown',
                'subdistrict': rel_parts[1] if len(rel_parts) > 1 else 'unknown',
                'station':     rel_parts[2] if len(rel_parts) > 2 else 'unknown',
            })
    results.sort(key=lambda x: (x['district'], x['subdistrict'], x['station']))
    return results


pdf_list = find_election_day_pdfs(SOURCE_DIR)
print(f'Total election-day PDFs found: {len(pdf_list)}')
print('\nFirst 3 entries:')
for entry in pdf_list[:3]:
    print(f"  [{entry['district']} / {entry['subdistrict']} / {entry['station']}]")

Total election-day PDFs found: 60

First 3 entries:
  [อำเภอบ้านไร่ / ตำบลบ้านบึง / หน่วยเลือกตั้งที่ 1]
  [อำเภอบ้านไร่ / ตำบลบ้านบึง / หน่วยเลือกตั้งที่ 2]
  [อำเภอบ้านไร่ / ตำบลบ้านบึง / หน่วยเลือกตั้งที่ 3]


---
## Task 3: PDF-to-Image Conversion (PyMuPDF)

In [14]:
def pdf_to_images(pdf_path: str, dpi: int = PDF_DPI) -> list[Image.Image]:
    """
    Convert each page of a PDF to a PIL Image using PyMuPDF.
    DPI 200 gives good OCR accuracy without excessive memory use.
    """
    images = []
    scale = dpi / 72.0
    matrix = fitz.Matrix(scale, scale)
    with fitz.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=matrix)
            images.append(Image.open(io.BytesIO(pix.tobytes('png'))))
    return images


# Verify on first PDF
test_images = pdf_to_images(pdf_list[0]['path'])
print(f'Pages: {len(test_images)}')
for i, img in enumerate(test_images):
    print(f'  Page {i+1}: size={img.size}, mode={img.mode}')

Pages: 2
  Page 1: size=(1653, 2339), mode=RGB
  Page 2: size=(1653, 2339), mode=RGB


---
## Task 4a: PaddleOCR Extraction (Priority 2)

In [15]:
import numpy as np

def run_paddle_ocr(images: list[Image.Image]) -> list[dict]:
    """
    Run PaddleOCR (PP-OCRv5) on a list of PIL Images.
    Returns per-page dicts: {page, full_text, blocks}
    """
    results = []
    for page_num, img in enumerate(images):
        img_array = np.array(img)
        raw = paddle_ocr.predict(img_array)  # returns list with one OCRResult per input

        blocks, lines = [], []
        if raw:
            res_data = raw[0].json.get('res', {})
            rec_texts  = res_data.get('rec_texts', [])
            rec_scores = res_data.get('rec_scores', [])
            rec_boxes  = res_data.get('rec_boxes', [])
            for text, score, bbox in zip(rec_texts, rec_scores, rec_boxes):
                blocks.append({'text': text, 'confidence': round(score, 4), 'bbox': bbox})
                lines.append(text)

        results.append({'page': page_num + 1, 'full_text': '\n'.join(lines), 'blocks': blocks})
    return results


# Run on sample
sample_pdfs = pdf_list[:SAMPLE_SIZE]
paddle_results = []

for i, entry in enumerate(sample_pdfs):
    print(f'\n--- PaddleOCR PDF {i+1}/{SAMPLE_SIZE}: {entry["station"]} ---')
    images = pdf_to_images(entry['path'])
    pages = run_paddle_ocr(images)
    paddle_results.append({'metadata': entry, 'pages': pages})

    print(f'Page 1 blocks (first 15):')
    for blk in pages[0]['blocks'][:15]:
        print(f'  [{blk["confidence"]:.2f}] {blk["text"]}')
    has_thai = any('\u0e00' <= c <= '\u0e7f' for c in pages[0]['full_text'])
    print(f'  Contains Thai: {has_thai}')

print('\nPaddleOCR complete.')


--- PaddleOCR PDF 1/3: หน่วยเลือกตั้งที่ 1 ---
Page 1 blocks (first 15):
  [0.94] ชุดที่ 1เก็บไว้ในเล่มเพื่อส่งคณะกรรมการการเลือกตั้งประจำเขตเลือกตั้ง (แบบแบ่งเขตเลือุกตั้ง)
  [0.93] ส.ส. ๕/๑๘
  [0.99] สีขาว
  [0.99] รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง
  [0.94] ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง
  [0.20] y
  [0.83] ได้กำหนดให้วันที... เดือน.พ.ศ.
  [0.46] 256a
  [0.93] เป็นวันเลือกตั้งนั้น
  [0.95] บัดนี้ คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขต
  [0.89] เลือกตั้งของหน่วยเลือกตั้งที่.
  [0.83] ..ตำบล/แขวง/เทศขาล..
  [0.33] mhm
  [0.81] อำเภอ/เยต
  [0.16] h
  Contains Thai: True

--- PaddleOCR PDF 2/3: หน่วยเลือกตั้งที่ 2 ---
Page 1 blocks (first 15):
  [0.96] ชุดที่  เก็บไว้ในเล่มเพื่อส่งคณะกรรมการการเลือกตั้งประจำเขตเลือกตั้ง (แบบแบ่งเขตเลือกตั้ง)
  [0.98] ส.ส. ๕/๑๘
  [0.99] สีขาว
  [0.97] รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง
  [0.95] ตามที่ได้มีพระร

---
## Task 4b: Tesseract Extraction (Priority 3)

In [16]:
def run_tesseract_ocr(images: list[Image.Image], lang: str = 'tha+eng') -> list[dict]:
    """
    Run Tesseract on a list of PIL Images.
    Returns per-page dicts: {page, full_text}
    """
    results = []
    for page_num, img in enumerate(images):
        text = pytesseract.image_to_string(img, lang=lang)
        results.append({'page': page_num + 1, 'full_text': text.strip()})
    return results


tesseract_results = []

for i, entry in enumerate(sample_pdfs):
    print(f'\n--- Tesseract PDF {i+1}/{SAMPLE_SIZE}: {entry["station"]} ---')
    images = pdf_to_images(entry['path'])
    pages = run_tesseract_ocr(images)
    tesseract_results.append({'metadata': entry, 'pages': pages})

    preview = pages[0]['full_text'][:500].replace('\n', ' ')
    print(f'  Page 1 preview: {preview}')
    has_thai = any('\u0e00' <= c <= '\u0e7f' for c in pages[0]['full_text'])
    print(f'  Contains Thai: {has_thai}')

print('\nTesseract complete.')


--- Tesseract PDF 1/3: หน่วยเลือกตั้งที่ 1 ---
  Page 1 preview: ชุดที่ ๑ เก็บไว้ในเล่มเพื่อส่งคณะกรรมการการเลือกตั้งประจําเขตเลือกตั้ง (แบบแบ่งเขตเลือก )                                        ส.สี. ๕/๑๕  สีขาว  รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง  ตามทีได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง eae onus. ad     ด                    ศรส ได้กําหนดให้วันที่ ..% ...... เดือน สละณัณ์พศ.  25%3 . เป็นวันเลือกตั้ง นั้น  vt               ยเลือกลังได้สกาคคาจนช               »  บัดนี คณะกรรมการประจําหน่วยเล
  Contains Thai: True

--- Tesseract PDF 2/3: หน่วยเลือกตั้งที่ 2 ---
  Page 1 preview: ขุดที่ ๑ เก็บไว้ในเล่มเพื่อส่งคณะกรรมการการเลือกตั้งประจําเขตเลือกตั้ง ล กๆ ก  ได้กําหนดให้วั   ii.                                                   ร. บัดนี้ นี คณะกรรมการประจําหน่วยเลอกตั้งไดดําเป็การนับคะแบบสมาชิกสภาผู้เหนราชฎรแบบแบปงต  เขตเลือกตั้งที่ ...๒... จังหวัดอุทัยธานี เสร็จสิ้นเป็นที่เรียบร้อยแล้ว & ดังนั้น จึงขอรายงานผลการนับคะแน

---
## Task 4c: Typhoon OCR Extraction (Priority 1)

Typhoon OCR is a Thai-specialist API model. We call it directly on the PDF file
using the `typhoon-ocr` package — no manual image conversion needed.
Requires `TYPHOON_OCR_API_KEY` in `.env`.

In [17]:
def crop_pdf_page_to_image(
    pdf_path: str,
    page_num: int = 1,
    dpi: int = 200,
    crop_top_ratio: float = 0.35,
) -> str:
    """Render a PDF page, crop to the data region (skip header), save as temp JPEG.

    crop_top_ratio=0.35 skips the title/preamble text (top 35%),
    keeping the vote-count statistics and candidate table.
    Returns the temp file path (caller must delete it).
    """
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
    doc.close()

    top = int(img.height * crop_top_ratio)
    cropped = img.crop((0, top, img.width, img.height))

    tmp = tempfile.NamedTemporaryFile(suffix='.jpg', delete=False)
    cropped.save(tmp.name, 'JPEG', quality=95)
    tmp.close()
    return tmp.name


def run_typhoon_ocr(pdf_path: str, page_num: int = 1, crop_top_ratio: float = 0.35) -> str:
    """Crop the PDF page to the data region, then run Typhoon OCR on the crop.

    Skips the top crop_top_ratio of the page (header/title) to send a smaller
    image, reducing API latency and keeping the model focused on the
    vote-count statistics and candidate table.
    """
    cropped_path = crop_pdf_page_to_image(pdf_path, page_num=page_num, crop_top_ratio=crop_top_ratio)
    try:
        result = ocr_document(
            pdf_or_image_path=cropped_path,
            figure_language='Thai',
        )
    finally:
        os.unlink(cropped_path)
    return result


typhoon_results = []

if not os.getenv('TYPHOON_OCR_API_KEY'):
    print('TYPHOON_OCR_API_KEY not set — skipping Typhoon OCR.')
    print('Set it in .env and rerun this cell.')
else:
    for i, entry in enumerate(sample_pdfs):
        station = entry['station']
        print(f'\n--- Typhoon OCR PDF {i+1}/{SAMPLE_SIZE}: {station} ---')
        try:
            text = run_typhoon_ocr(entry['path'], page_num=1)
            typhoon_results.append({'metadata': entry, 'page1_text': text})
            preview = text[:600].replace('\n', ' ')
            print(f'  Page 1 preview: {preview}')
            has_thai = any('\u0e00' <= ch <= '\u0e7f' for ch in text)
            print(f'  Contains Thai: {has_thai}')
        except Exception as e:
            print(f'  ERROR: {e}')
            typhoon_results.append({'metadata': entry, 'page1_text': '', 'error': str(e)})

    print(f'\nTyphoon OCR complete: {len(typhoon_results)} PDFs processed.')



--- Typhoon OCR PDF 1/3: หน่วยเลือกตั้งที่ 1 ---


[2026-04-09 20:06:37,965] [    INFO] _client.py:1025 - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"


  Page 1 preview: ๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน (เฉพาะวันเลือกตั้ง) จำนวน 295 คน (สองร้อยเก้าสิบห้า) ๒. จำนวนบัตรเลือกตั้ง ๒.๑ จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร จำนวน A20 บัตร (สี่พันสองร้อยห้าสิบห้า) ๒.๒ จำนวนบัตรเลือกตั้งที่ใช้ จำนวน 295 บัตร (สองร้อยเก้าสิบห้า) ๒.๒.๑ บัตรดี จำนวน 277 บัตร (สองร้อยเจ็ดสิบห้า) ๒.๒.๒ บัตรเสีย จำนวน 6 บัตร (ห้า) ๒.๒.๓ บัตรที่ไม่เลือกผู้สมัครผู้ใด จำนวน 12 บัตร (สิบสอง) ๒.๓ จำนวนบัตรเลือกตั้งที่เหลือ จำนวน 125 บัตร (หนึ่งร้อยยี่สิบห้า)  ๓. จำนวนคะแนนที่ผู้สมัครรับเลือกตั้งแต่ละคนได้รับเรียงตามลำดับหมายเลขประจำตัวผู้สมัคร  <table><tr><td>หมายเลขประจำตัว ผู้สมัคร</td><td>
  Contains Thai: True

--- Typhoon OCR PDF 2/3: หน่วยเลือกตั้งที่ 2 ---


[2026-04-09 20:09:40,126] [    INFO] _client.py:1025 - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 408 Request Timeout"
[2026-04-09 20:09:40,127] [    INFO] _base_client.py:1111 - Retrying request to /chat/completions in 0.417380 seconds
[2026-04-09 20:12:42,697] [    INFO] _client.py:1025 - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 408 Request Timeout"
[2026-04-09 20:12:42,698] [    INFO] _base_client.py:1111 - Retrying request to /chat/completions in 0.961069 seconds
[2026-04-09 20:15:45,681] [    INFO] _client.py:1025 - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 408 Request Timeout"


  ERROR: Error code: 408 - {'detail': 'Request timed out: An error occurred during model processing'}

--- Typhoon OCR PDF 3/3: หน่วยเลือกตั้งที่ 3 ---


[2026-04-09 20:15:50,282] [    INFO] _client.py:1025 - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"


  Page 1 preview: ๑.๑ จำนวนผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อผู้มีสิทธิเลือกตั้ง จำนวน 222 คน (สองร้อยยี่สิบสอง) ๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน (เฉพาะวันเลือกตั้ง) จำนวน 169 คน (หนึ่งร้อยหกสิบเก้า)  ๒. จำนวนบัตรเลือกตั้ง ๒.๑ จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร จำนวน 240 บัตร (สองร้อยสี่สิบ) ๒.๒ จำนวนบัตรเลือกตั้งที่ใช้ จำนวน 167 บัตร (หนึ่งร้อยหกสิบเก้า) (ลงชื่อ)  ๒.๒.๑ บัตรดี จำนวน 163 บัตร (หนึ่งร้อยหกสิบสาม) (ลงชื่อ)  ๒.๒.๒ บัตรเสีย จำนวน 8 บัตร (สาม) (สำหรับระบุเหตุผล) ๒.๒.๓ บัตรที่ไม่เลือกผู้สมัครผู้ใด จำนวน 3 บัตร (สาม) ๒.๓ จำนวนบัตรเลือกตั้งที่เหลือ จำนวน 71 บัตร (เจ็ดสิบเอ็ด)  ๓. จำนวนคะแนนที่ผู้
  Contains Thai: True

Typhoon OCR complete: 3 PDFs processed.


---
## Task 5: Side-by-Side Comparison & Regex Extraction

Compare all three engines on the same station. The regex extractor runs on Typhoon OCR
output by default (Priority 1) and falls back to PaddleOCR if Typhoon is unavailable.

In [18]:
THAI_DIGIT_MAP = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

def thai_to_arabic(text: str) -> str:
    return text.translate(THAI_DIGIT_MAP)


def extract_station_data(ocr_text: str) -> dict:
    """
    Extract key structured fields from full-page OCR text of a \u0e2a.\u0e2a.5/18 form.
    Applies regex template matching against known keyword anchors.
    """
    converted = thai_to_arabic(ocr_text)

    result = {
        'constituency_number': None,
        'station_number': None,
        'form_type_detected': False,
        'section_labels_found': [],
        'party_names': [],
        'candidate_names': [],
    }

    # Form type: ส.ส. 5/18
    if re.search(r'\u0e2a\.\u0e2a\.\s*[5\u0e55]/[18\u0e51\u0e58]', converted, re.IGNORECASE):
        result['form_type_detected'] = True

    # Constituency number
    m = re.search(r'\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48[^\d]*(\d+)', converted)
    if m:
        result['constituency_number'] = int(m.group(1))

    # Station number
    m = re.search(r'\u0e2b\u0e19\u0e48\u0e27\u0e22\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48[^\d]*(\d+)', converted)
    if m:
        result['station_number'] = int(m.group(1))

    # Section labels
    known_sections = {
        '\u0e1c\u0e39\u0e49\u0e21\u0e35\u0e2a\u0e34\u0e17\u0e18\u0e34\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07': '1.1_eligible_voters',
        '\u0e1a\u0e31\u0e15\u0e23\u0e14\u0e35': '2.2.1_valid_ballots',
        '\u0e1a\u0e31\u0e15\u0e23\u0e40\u0e2a\u0e35\u0e22': '2.2.2_spoiled_ballots',
        '\u0e23\u0e27\u0e21\u0e04\u0e30\u0e41\u0e19\u0e19': '3_total_score',
        '\u0e04\u0e30\u0e41\u0e19\u0e19\u0e17\u0e35\u0e48': '3_candidate_scores',
    }
    for keyword, label in known_sections.items():
        if keyword in ocr_text:
            result['section_labels_found'].append(label)

    # Known parties
    known_parties = [
        '\u0e40\u0e1e\u0e37\u0e48\u0e2d\u0e44\u0e17\u0e22',
        '\u0e20\u0e39\u0e21\u0e34\u0e43\u0e08\u0e44\u0e17\u0e22',
        '\u0e1b\u0e23\u0e30\u0e0a\u0e32\u0e0a\u0e19',
        '\u0e1b\u0e23\u0e30\u0e0a\u0e32\u0e18\u0e34\u0e1b\u0e31\u0e15\u0e22\u0e4c',
        '\u0e01\u0e25\u0e49\u0e32\u0e18\u0e23\u0e23\u0e21',
        '\u0e44\u0e17\u0e22\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e44\u0e17\u0e22',
        '\u0e23\u0e27\u0e21\u0e44\u0e17\u0e22\u0e2a\u0e23\u0e49\u0e32\u0e07\u0e0a\u0e32\u0e15\u0e34',
    ]
    result['party_names'] = [p for p in known_parties if p in ocr_text]

    # Candidate names (นาย / นางสาว / นาง prefix)
    names = re.findall(
        r'(?:\u0e19\u0e32\u0e22|\u0e19\u0e32\u0e07\u0e2a\u0e32\u0e27|\u0e19\u0e32\u0e07)[^\n]+',
        ocr_text
    )
    result['candidate_names'] = [n.strip() for n in names if len(n.strip()) > 5]

    return result


# --- Side-by-side comparison ---
print('=' * 60)
print('SIDE-BY-SIDE OCR COMPARISON (Station 1)')
print('=' * 60)

first_paddle   = paddle_results[0]['pages'][0]['full_text'] if paddle_results else ''
first_tesseract = tesseract_results[0]['pages'][0]['full_text'] if tesseract_results else ''
first_typhoon  = typhoon_results[0]['page1_text'] if typhoon_results else ''

print(f'\n[PaddleOCR - first 400 chars]')
print(first_paddle[:400])

print(f'\n[Tesseract - first 400 chars]')
print(first_tesseract[:400])

print(f'\n[Typhoon OCR - first 400 chars]')
print(first_typhoon[:400] if first_typhoon else '(not run — API key not set)')

# --- Extraction using best available source ---
best_text = first_typhoon or first_paddle
print(f'\n[Regex extraction on: {"Typhoon" if first_typhoon else "PaddleOCR"}]')
extracted = extract_station_data(best_text)
for k, v in extracted.items():
    print(f'  {k}: {v}')

SIDE-BY-SIDE OCR COMPARISON (Station 1)

[PaddleOCR - first 400 chars]
ชุดที่ 1เก็บไว้ในเล่มเพื่อส่งคณะกรรมการการเลือกตั้งประจำเขตเลือกตั้ง (แบบแบ่งเขตเลือุกตั้ง)
ส.ส. ๕/๑๘
สีขาว
รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง
ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง
y
ได้กำหนดให้วันที... เดือน.พ.ศ.
256a
เป็นวันเลือกตั้งนั้น
บัดนี้ คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขต

[Tesseract - first 400 chars]
ชุดที่ ๑ เก็บไว้ในเล่มเพื่อส่งคณะกรรมการการเลือกตั้งประจําเขตเลือกตั้ง (แบบแบ่งเขตเลือก
)                                        ส.สี. ๕/๑๕

สีขาว

รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง

ตามทีได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง
eae onus. ad     ด                    ศรส
ได้กําหนดให้วันที่ ..% ...... เดือน สละณัณ์พศ.  25%3 . เป็นวันเ

[Typhoon OCR - first 400 chars]
๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน (เฉพาะวันเลือกตั้ง) จำน

---
## Prototype Findings Summary

### Engine Comparison
| Engine | Thai printed text | Handwritten numbers | Setup |
|--------|------------------|---------------------|-------|
| Typhoon OCR | Expected: excellent | Expected: good (specialist) | API key required |
| PaddleOCR | Good | Poor (garbled) | Local |
| Tesseract | Fair | Poor | Local |

### Phase 2 Plan
1. Use Typhoon OCR as primary engine for handwritten vote count fields.
2. Use PaddleOCR as fallback for structural text when API is unavailable.
3. Extract station metadata from directory path (reliable, engine-independent).
4. Scale to all 60 polling stations and export `election_structured_data.csv`.

### Requirements Addressed
- **EXTR-01**: PDF-to-image conversion implemented (PyMuPDF, 200 DPI)
- **EXTR-02**: Directory traversal finds all ส.ส.5/18 PDFs
- **EXTR-03**: Thai text extracted (PaddleOCR + Tesseract + Typhoon OCR)
- **EXTR-04**: All three engines tested in priority order (Typhoon > PaddleOCR > Tesseract)